In [152]:
import oci

# Cargar configuración
config = oci.config.from_file("~/.oci/config", "DEFAULT")

# Crear cliente del servicio de "Management"
client = oci.generative_ai.GenerativeAiClient(config)

# Usa tu compartment_idid
compartment_id = "ocid1.tenancy.oc1..aaaaaaaajviqbmt2sli22ow3c3oiz2ti5q4vqgacdlj47ujjo3exfkdi45wa"

# Listar los modelos disponibles en tu región y compartment
response = client.list_models(compartment_id=compartment_id)

#for model in response.data.items:
#    print(f"{model} - {model.display_name}")


In [155]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Carga todos los Json de la carpeta data/
loader = DirectoryLoader(
    "data",
    glob="*.json",
    loader_cls=JSONLoader,
    loader_kwargs={
        "jq_schema": ".promociones[] | tostring",
        "text_content": True,
    },
)
documents = loader.load()

# Divide en fragmentos manejables
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = splitter.split_documents(documents)
print(f"Total de fragmentos: {len(texts)}")


Total de fragmentos: 37


Prueba con embeddings de OCI

In [156]:
from langchain_oci.embeddings import OCIGenAIEmbeddings

embeddings = OCIGenAIEmbeddings(
    model_id="cohere.embed-multilingual-light-v3.0",
    service_endpoint="https://inference.generativeai.us-chicago-1.oci.oraclecloud.com",
    compartment_id="ocid1.tenancy.oc1..aaaaaaaajviqbmt2sli22ow3c3oiz2ti5q4vqgacdlj47ujjo3exfkdi45wa",
)


In [157]:
from langchain_community.vectorstores import Chroma
import os

BASE_DIR = "/mnt/data"
if not os.path.exists(BASE_DIR):
    # fallback to your home dir
    BASE_DIR = os.path.expanduser("~")

CHROMA_DIR = os.path.join(BASE_DIR, "chroma_db")
os.makedirs(CHROMA_DIR, exist_ok=True)

# 2. quick sanity check: can we write here?
test_path = os.path.join(CHROMA_DIR, "write_test.txt")
with open(test_path, "w") as f:
    f.write("ok")

# 3. now build the vectorstore
vectorstore = Chroma.from_documents(
    texts,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
)
vectorstore.persist()


In [158]:
from langchain_oci.chat_models import ChatOCIGenAI

llm = ChatOCIGenAI(
    model_id="ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceya3zoyev5tgdo3puutjfmxfnmpjutihhgqgtbyr7q6qtja",  # o llama-3.x según lo que tengas habilitado
    compartment_id="ocid1.tenancy.oc1..aaaaaaaajviqbmt2sli22ow3c3oiz2ti5q4vqgacdlj47ujjo3exfkdi45wa",
    service_endpoint="https://inference.generativeai.us-chicago-1.oci.oraclecloud.com",
)


In [159]:
from langchain.chains import RetrievalQA

SYSTEM_PROMPT = """
Eres un asistente de promociones en la fase de pago de un e-commerce de ropa en México.
Tienes una lista de promociones estructuradas. 
Con base en:
- método de pago (tarjeta, banco, etc.)
- monto de la compra en MXN
elige la promoción más adecuada.
Responde en tono persuasivo, corto y claro, mencionando el beneficio y las condiciones principales.
Si no hay promoción aplicable, di que no hay, pero sugiere subir un poco el monto si aplica.
"""

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": None,  # some versions need a prompt template; showing idea
    },
)


In [181]:
from datetime import datetime, timezone
import json

def promotor(monto=0, banco=""):
    hoy = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    query = f"""
    Fecha actual (UTC): {hoy}
    Cliente pagando {monto} MXN con tarjeta {banco}.

    Tienes una lista de promociones. Debes hacer DOS cosas:

    1) PROMO_ACTUAL: elegir la mejor promoción que:
       - sea del banco {banco}
       - esté vigente (vigencia_inicio <= fecha actual <= vigencia_fin o fin = null)
       - y cuyo minimo_carrito_mxn sea <= {monto}
       Si no hay ninguna que cumpla todo eso, pon "promo_title": "Sin promoción" y "meets_minimum": false.

    2) PROXIMA_PROMO: de las promociones del MISMO banco {banco} y vigentes,
       elige la que NO cumpla el monto (es decir, cuyo minimo_carrito_mxn sea > {monto}),
       pero que tenga el minimo_carrito_mxn más cercano por ARRIBA.
       Esto es para sugerirle al cliente que suba su compra.
       Si no hay una más arriba, deja este objeto vacío.

    RESPONDE SOLO en ESTE JSON EXACTO:

    {{
      "current_promo": {{
        "promo_title": "nombre corto de la promo o 'Sin promoción'",
        "message": "mensaje persuasivo para mostrarle al cliente en checkout",
        "meets_minimum": true
      }},
      "next_promo": {{
        "promo_title": "nombre corto de la promo más cercana por arriba o '' si no hay",
        "required_amount": 0,
        "message": "mensaje persuasuivo y amigable diciendo cuánto le falta para alcanzarla o '' si no hay"
      }}
      "mix_message": {{
        "message": "reescribe los mensajes de current_promo y next_promo para que quede un texto logico y amigable y como si fueras vendedor profesional de ropa sin inventar cosas que no sabes si son verdad, se más directo sin saludo"
      }}
    }}
    """
    result = qa_chain.invoke(query)

    # filtrar fuentes por banco
    valid_sources = []
    for doc in result["source_documents"]:
        try:
            promo = json.loads(doc.page_content)
        except Exception:
            continue

        bancos_promo = promo.get("condiciones", {}).get("bancos", [])
        # si el user pasó banco="", no aceptamos nada
        if not banco:
            continue
        # bancos pueden venir como ["BBVA", "Santander"]
        if banco in bancos_promo:
            valid_sources.append(promo)

    if not banco or not valid_sources:
        return json.dumps({
            "promo_title": "Sin promoción",
            "message": "No hay promociones para ese banco en este momento.",
            "meets_minimum": False
        }, ensure_ascii=False)

    # si sí hubo, dejamos la respuesta del modelo
    return result["result"]

result = promotor(monto=500, banco="BBVA")
print(result)

{
  "current_promo": {
    "promo_title": "5% BBVA desde $500",
    "message": "Paga con tu tarjeta BBVA y obtén 5% de descuento en compras desde $500.",
    "meets_minimum": true
  },
  "next_promo": {
    "promo_title": "8% BBVA desde $900",
    "required_amount": 900,
    "message": "Sube tu compra a $900 para obtener 8% de descuento con BBVA, te faltan solo $400."
  },
  "mix_message": {
    "message": "Con tu compra de $500 y tarjeta BBVA, obtén 5% de descuento ahora mismo. Si subes a $900, llegarás al 8% de descuento, te faltan solo $400 para mejorarlo."
  }
}
